In [ ]:
from hw2fintools import gurufocus as gf
import os
from dotenv import load_dotenv
import pandas as pd
import re


load_dotenv()
path_stockdata = os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList')
date_pattern = r"(\d\d\d\d-\d\d-\d\d)--"


In [ ]:
ticker = input('Enter ticker symbol: ')
print('Ticker is: ', ticker)

In [ ]:
path_ticker = os.path.join(path_stockdata, ticker.upper())
div_list = []

for item in os.listdir(path_ticker):
    if 'gf-raw-dividend_history' in item and item.endswith('.csv') and not item.startswith('._'):
        div_list.append(item)

div_list_sorted = sorted(div_list, reverse=True)
div_current = div_list_sorted[0]

div_path = os.path.join(path_ticker, div_current)
div_path

In [ ]:
price_list = []

for item in os.listdir(path_ticker):
    if 'gf-raw-price_history' in item and item.endswith('.csv') and not item.startswith('._'):
        price_list.append(item)

price_list_sorted = sorted(price_list, reverse=True)
price_current = price_list_sorted[0]

price_path = os.path.join(path_ticker, price_current)
price_path

In [ ]:
div_date = re.search(date_pattern, div_path)
price_date = re.search(date_pattern, price_path)

div_match = div_date.group(1)
price_match = price_date.group(1)

if div_match == price_match:
    print('date match')
else:
    print('date mismatch')


In [ ]:
div_df0 = gf.div_hist_s1v1(div_path)


In [ ]:
div_df1 = div_df0.loc[div_df0['DivType'] == 'regular']
div_df1 = div_df1.drop(columns=['DivRecordDate', 'DivDeclareDate', 'DivPayDate'])
div_df1 = div_df1.rename(columns={'ExDivDate': 'Date'})
div_df1

In [ ]:
price_df0 = gf.price_hist_s2v1(price_path)

In [ ]:
merged_df0 = pd.merge(price_df0, div_df1, on='Date', how='left')
merged_df0['DivAmount'] = merged_df0['DivAmount'].fillna(0)
merged_df0['DivFrequency'] = merged_df0['DivFrequency'].fillna(div_df1.iloc[0]['DivFrequency'])
merged_df0['DivType'] = merged_df0['DivType'].fillna(div_df1.iloc[0]['DivType'])
merged_df0['DivPayDeclared'] = merged_df0['DivAmount'].fillna(0)
merged_df0

In [ ]:
div_var = 0

for index, row in merged_df0.iterrows():
    if row['DivAmount'] > 0:
        div_var = row['DivAmount']

    else:
        merged_df0.at[index, 'DivAmount'] = div_var


merged_df0['FwdDiv'] = merged_df0['DivFrequency'] * merged_df0['DivAmount']
merged_df0['FwdDivYield'] = merged_df0['FwdDiv'] / merged_df0['PricePerShare']

merged_df0

In [ ]:
aggr_df0 = merged_df0
aggr_df0 = aggr_df0.set_index('Date')
aggr_df1 = aggr_df0.groupby(aggr_df0.index.year).agg(
    SharePriceMin=pd.NamedAgg(column='PricePerShare', aggfunc='min'),
    SharePriceMax=pd.NamedAgg(column='PricePerShare', aggfunc='max'),
    SharePriceMean=pd.NamedAgg(column='PricePerShare', aggfunc='mean'),
    SharePriceMedian=pd.NamedAgg(column='PricePerShare', aggfunc='median'),
    DivYieldMin=pd.NamedAgg(column='FwdDivYield', aggfunc='min'),
    DivYieldMax=pd.NamedAgg(column='FwdDivYield', aggfunc='max'),
    DivYieldMean=pd.NamedAgg(column='FwdDivYield', aggfunc='mean'),
    DivYieldMedian=pd.NamedAgg(column='FwdDivYield', aggfunc='median'),
    DivPaidTotal=pd.NamedAgg('DivPayDeclared', aggfunc='sum')
)

aggr_df1.index.name = 'DateCy'
aggr_df1

In [ ]:
merged_df0.to_csv(os.path.join(path_ticker, ticker.upper() + '--DivPrice_History--s3v1.csv'))

In [ ]:
aggr_df1.to_csv(os.path.join(path_ticker, ticker.upper() + '--Aggregate_Cy_DivPrice_History--s4v1.csv'))

In [ ]:
aggr_df1.index